# Linear span-reranker report

Upload or locate `linear_span_reranker_artifacts.zip`. This notebook performs no training or inference and does not require the dataset.

## Step 1 — Load and validate the artifact


In [1]:
import json,csv,io,zipfile,sys,collections
from pathlib import Path
from IPython.display import display,Markdown
NAME="linear_span_reranker_artifacts.zip"
paths=[Path(NAME),Path("/content")/NAME,Path("/content/drive/MyDrive/CITS4012_A2_reranker")/NAME]
archive=next((p for p in paths if p.exists()),None)
if archive is None and "google.colab" in sys.modules:
    from google.colab import drive; drive.mount("/content/drive"); archive=next((p for p in paths if p.exists()),None)
if archive is None and "google.colab" in sys.modules:
    from google.colab import files
    up=files.upload(); assert NAME in up; archive=Path("/content")/NAME; archive.write_bytes(up[NAME])
assert archive is not None
zf=zipfile.ZipFile(archive); assert zf.testzip() is None
summary=json.loads(zf.read("reranker_results/reranker_summary.json")); rows=list(csv.DictReader(io.StringIO(zf.read("reranker_results/paired_deltas.csv").decode())))
print("Loaded",archive,"source",summary["source_run"],"— no training or inference")


Loaded linear_span_reranker_artifacts.zip source CurrentGated-s1-20261014-175436 — no training or inference


## Step 2 — Compare reranker metrics


In [2]:
metrics=("bleu1","meteor","rougeL","em","f1"); lines=["| System | BLEU-1 | METEOR | ROUGE-L | EM | F1 |","|---|---:|---:|---:|---:|---:|"]
for name in ("BaseTop1","LinearReranker","LambdaMART","Top10Oracle"):
    v=summary["scores"][name]; lines.append(f"| {name} | "+" | ".join(f"{v[m]:.2f}" for m in metrics)+" |")
display(Markdown("## Validation results\n\n"+"\n".join(lines)))
for name in ("LinearReranker","LambdaMART"):
    d=summary["delta_vs_base"][name]; display(Markdown(f"**{name} − base:** "+", ".join(f"{m} {d[m]:+.2f}" for m in metrics)))


## Validation results

| System | BLEU-1 | METEOR | ROUGE-L | EM | F1 |
|---|---:|---:|---:|---:|---:|
| BaseTop1 | 39.06 | 33.69 | 44.14 | 27.32 | 44.22 |
| LinearReranker | 39.15 | 32.38 | 44.10 | 28.53 | 44.40 |
| LambdaMART | 39.46 | 33.19 | 44.51 | 27.70 | 44.69 |
| Top10Oracle | 62.02 | 51.33 | 67.16 | 49.16 | 67.15 |

**LinearReranker − base:** bleu1 +0.09, meteor -1.31, rougeL -0.04, em +1.21, f1 +0.18

**LambdaMART − base:** bleu1 +0.40, meteor -0.49, rougeL +0.37, em +0.37, f1 +0.47

## Step 3 — Run paired bootstrap analysis


In [3]:
import numpy as np
rng=np.random.default_rng(4012); lines=["| Model vs base | Metric | Mean paired Δ | 95% bootstrap CI | P(Δ≤0) | Win / tie / loss |","|---|---|---:|---:|---:|---:|"]
for prefix,label in (("linear","Linear"),("tree","LambdaMART")):
 for metric in ("rougeL","f1","em"):
    d=np.array([float(r[f"{prefix}_{metric}"]) for r in rows]); means=np.array([d[rng.integers(0,len(d),len(d))].mean() for _ in range(10000)])*100
    lo,hi=np.quantile(means,[.025,.975]); lines.append(f"| {label} | {metric} | {d.mean()*100:+.2f} | [{lo:+.2f}, {hi:+.2f}] | {(means<=0).mean():.3f} | {(d>1e-12).sum()} / {(abs(d)<=1e-12).sum()} / {(d < -1e-12).sum()} |")
display(Markdown("## Paired bootstrap\n\n"+"\n".join(lines)))


## Paired bootstrap

| Model vs base | Metric | Mean paired Δ | 95% bootstrap CI | P(Δ≤0) | Win / tie / loss |
|---|---|---:|---:|---:|---:|
| Linear | rougeL | -0.04 | [-0.71, +0.64] | 0.553 | 29 / 1008 / 39 |
| Linear | f1 | +0.18 | [-0.48, +0.87] | 0.309 | 30 / 1008 / 38 |
| Linear | em | +1.21 | [+0.56, +1.95] | 0.000 | 14 / 1061 / 1 |
| LambdaMART | rougeL | +0.37 | [-0.38, +1.10] | 0.168 | 44 / 994 / 38 |
| LambdaMART | f1 | +0.47 | [-0.27, +1.22] | 0.113 | 45 / 994 / 37 |
| LambdaMART | em | +0.37 | [-0.46, +1.21] | 0.229 | 13 / 1054 / 9 |

## Step 4 — Inspect slices and selected ranks


In [4]:
for field,title in (("question_type","Question type"),("silver_region","Silver-span region")):
    groups=collections.defaultdict(list)
    for r in rows: groups[r[field]].append(r)
    lines=[f"| {title} | N | Linear ΔROUGE-L | LambdaMART ΔROUGE-L |","|---|---:|---:|---:|"]
    for name,v in sorted(groups.items()): lines.append(f"| {name} | {len(v)} | {sum(float(x['linear_rougeL']) for x in v)*100/len(v):+.2f} | {sum(float(x['tree_rougeL']) for x in v)*100/len(v):+.2f} |")
    display(Markdown(f"## {title} effects\n\n"+"\n".join(lines)))
for model in ("LinearReranker","LambdaMART"):
 rank_counts={int(k):v for k,v in summary["selected_rank_counts"][model].items()}; total=sum(rank_counts.values())
 display(Markdown(f"## {model} selected base ranks\n\n"+"\n".join(["| Rank | Count | Share |","|---:|---:|---:|"]+[f"| {k} | {rank_counts.get(k,0)} | {100*rank_counts.get(k,0)/total:.1f}% |" for k in range(1,11)])))


## Question type effects

| Question type | N | Linear ΔROUGE-L | LambdaMART ΔROUGE-L |
|---|---:|---:|---:|
| how | 85 | -0.83 | +0.46 |
| other | 95 | -0.01 | -1.28 |
| what | 462 | -0.30 | +0.42 |
| when | 24 | +0.79 | +2.41 |
| where | 75 | -1.47 | +0.76 |
| which | 22 | +0.00 | +0.00 |
| who | 290 | +1.25 | +0.93 |
| why | 23 | -4.52 | -4.31 |

## Silver-span region effects

| Silver-span region | N | Linear ΔROUGE-L | LambdaMART ΔROUGE-L |
|---|---:|---:|---:|
| body | 820 | +0.07 | +0.25 |
| byline | 76 | -1.33 | +2.48 |
| cross | 18 | +0.00 | +0.00 |
| no_span | 162 | +0.00 | +0.00 |

## LinearReranker selected base ranks

| Rank | Count | Share |
|---:|---:|---:|
| 1 | 980 | 91.1% |
| 2 | 80 | 7.4% |
| 3 | 10 | 0.9% |
| 4 | 5 | 0.5% |
| 5 | 1 | 0.1% |
| 6 | 0 | 0.0% |
| 7 | 0 | 0.0% |
| 8 | 0 | 0.0% |
| 9 | 0 | 0.0% |
| 10 | 0 | 0.0% |

## LambdaMART selected base ranks

| Rank | Count | Share |
|---:|---:|---:|
| 1 | 950 | 88.3% |
| 2 | 81 | 7.5% |
| 3 | 25 | 2.3% |
| 4 | 9 | 0.8% |
| 5 | 4 | 0.4% |
| 6 | 3 | 0.3% |
| 7 | 1 | 0.1% |
| 8 | 0 | 0.0% |
| 9 | 2 | 0.2% |
| 10 | 1 | 0.1% |

## Step 5 — Inspect feature effects


In [5]:
lines=["| Feature | Standardized coefficient |","|---|---:|"]+[f"| {name} | {weight:+.3f} |" for name,weight in summary["feature_weights"]]
display(Markdown("## Linear model weights\n\n"+"\n".join(lines)+"\n\nCoefficients are in standardized pair-difference space; magnitude indicates influence, not causality."))
lines=["| Feature | LambdaMART split importance |","|---|---:|"]+[f"| {name} | {weight} |" for name,weight in summary["tree_importances"]]
display(Markdown("## LambdaMART feature importances\n\n"+"\n".join(lines)+"\n\nSplit counts indicate usage, not causal effect."))


## Linear model weights

| Feature | Standardized coefficient |
|---|---:|
| base_score | +1.234 |
| score_margin | +1.234 |
| start_logp | +1.034 |
| end_logp | +0.770 |
| question_overlap | -0.532 |
| length_fraction | -0.401 |
| question_jaccard | +0.397 |
| ends_word | +0.283 |
| start_fraction | +0.281 |
| reciprocal_rank | +0.229 |
| context_overlap | +0.153 |
| contains_mention | -0.121 |
| contains_url | -0.114 |
| distance_from_end | +0.109 |
| end_fraction | -0.109 |
| length | -0.091 |
| starts_word | +0.078 |
| contains_number | +0.063 |
| region_byline | +0.053 |
| region_body | -0.026 |
| context_jaccard | +0.021 |
| region_cross | -0.017 |
| qtype_what | +0.000 |
| qtype_who | +0.000 |
| qtype_how | +0.000 |
| qtype_where | +0.000 |
| qtype_why | +0.000 |
| qtype_which | +0.000 |
| qtype_when | +0.000 |
| qtype_other | +0.000 |

Coefficients are in standardized pair-difference space; magnitude indicates influence, not causality.

## LambdaMART feature importances

| Feature | LambdaMART split importance |
|---|---:|
| length | 277 |
| base_score | 254 |
| start_logp | 251 |
| end_logp | 231 |
| score_margin | 219 |
| length_fraction | 147 |
| start_fraction | 115 |
| end_fraction | 90 |
| context_jaccard | 77 |
| distance_from_end | 70 |
| question_overlap | 59 |
| reciprocal_rank | 52 |
| context_overlap | 46 |
| question_jaccard | 37 |
| ends_word | 31 |
| starts_word | 24 |
| contains_mention | 21 |
| qtype_who | 16 |
| qtype_why | 13 |
| qtype_other | 11 |
| region_byline | 7 |
| region_body | 6 |
| region_cross | 3 |
| qtype_what | 2 |
| qtype_when | 2 |
| contains_number | 1 |
| qtype_how | 1 |
| qtype_where | 1 |
| contains_url | 0 |
| qtype_which | 0 |

Split counts indicate usage, not causal effect.

## Step 6 — Write the conclusion


In [6]:
best=max(("LinearReranker","LambdaMART"),key=lambda n:summary["scores"][n]["rougeL"]); d=summary["delta_vs_base"][best]; gain=d["rougeL"]
decision="supports" if gain>0 else "does not support"
text=f"""# Linear Top-10 span reranker

The stronger fixed reranker was **{best}**. It changed validation ROUGE-L by **{gain:+.2f}**, F1 by **{d['f1']:+.2f}**,
and EM by **{d['em']:+.2f}** versus the unchanged Top-1 decoder. This result **{decision}** reranking
with the current feature set. The reranker was fitted only from training-split labels; validation labels
were used only for evaluation. The locked test set was not accessed.
"""
Path("linear_span_reranker_report.md").write_text(text,encoding="utf-8"); display(Markdown("## Report-ready conclusion\n\n"+text)); zf.close()


## Report-ready conclusion

# Linear Top-10 span reranker

The stronger fixed reranker was **LambdaMART**. It changed validation ROUGE-L by **+0.37**, F1 by **+0.47**,
and EM by **+0.37** versus the unchanged Top-1 decoder. This result **supports** reranking
with the current feature set. The reranker was fitted only from training-split labels; validation labels
were used only for evaluation. The locked test set was not accessed.
